# Notebook 03: Tendinopathy Detection — CNN and Hybrid Model

This notebook covers the two classification modes used to detect rotator cuff tendinopathy
from shoulder X-rays: a pure convolutional approach (SEG_300) and a hybrid approach that
combines deep features with clinical variables (Fast_512).

**What you will learn:**
- Why VGG19 is a suitable backbone for medical image classification
- The SEG_300 mode: classify a 300x300 crop from the U-Net output
- The Fast_512 mode: classify a 512x512 full image using a combined CNN + ML pipeline
- How to extract intermediate feature maps using the GlobalMaxPooling layer
- How clinical variables (age, sex) are incorporated into the hybrid model
- How Grad-CAM produces spatial attention maps on the X-ray

**Source code references:**
- `Backend/ai_logic.py` — `pipeline_300()`, `classify_crop()`, `get_gradcam_image()`
- `Backend/model_loader.py` — CNN, combined_cnn, ml_model loading
- `Backend/app.py` — `/predict`, `/predict_512_fast`, `/predict_512_gradcam` endpoints

> **Note on model weights:** Inference cells require the CNN checkpoint (.h5) and ML model (.joblib)
> loaded via `Backend/config.py`. Cells will print a warning and skip if no weights are available.

## 1. VGG19 as a Transfer Learning Backbone

VGG19 (Simonyan and Zisserman, 2014) is a deep CNN with 19 weight layers (16 convolutional,
3 fully connected) trained on ImageNet. Despite being older than more recent architectures,
it remains widely used in medical imaging because:

- Its architecture is simple and interpretable (uniform 3x3 convolutions).
- The learned filters generalize well to grayscale X-ray textures when fine-tuned.
- Grad-CAM visualization is straightforward since `block5_conv4` is the last spatial layer.

### Transfer learning strategy
Starting from ImageNet weights (trained on natural color photographs) and fine-tuning on
X-rays is effective because low-level feature detectors (edges, textures, oriented filters)
are domain-agnostic. Only higher layers need to be adapted to the medical domain.

The model in this project is built on top of VGG19 with:
- A **GlobalMaxPooling2D** layer after the convolutional base, which collapses the spatial
  dimensions into a fixed-length feature vector (512 dimensions for VGG19).
- A single **Dense(1, activation='sigmoid')** output node for binary classification
  (tendinopathy / no tendinopathy).

GlobalMaxPooling retains the strongest activation across the entire feature map for each
channel, making it more robust to the position of the finding within the image compared
to GlobalAveragePooling.

## 2. Two Inference Modes

The application exposes two tendinopathy detection endpoints, each optimized for a
different clinical scenario:

### Mode A — SEG_300 (`/predict`)
```
Full image
   -> U-Net crop (Notebook 02)
   -> Resize to 300x300
   -> VGG19 classifier
   -> probability + diagnosis
```
This mode is the primary detection pathway. It focuses the classifier on the cropped
humeral head at a resolution of 300x300 pixels.

### Mode B — Fast_512 (`/predict_512_fast`)
```
Full image
   -> Resize to 512x512 (no crop)
   -> VGG19 (combined model)
       |-> probability (cnn_output)
       |-> 512-dim feature vector (GlobalMaxPooling)
   -> [features + age + sex] -> ML model
   -> probability + diagnosis
```
This mode adds clinical context. The CNN features are augmented with two tabular variables
(patient age and sex) and passed to a trained scikit-learn pipeline for a final prediction.
Both the raw CNN probability and the hybrid probability are returned.

## 3. Imports and Setup

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

try:
    import tf_keras
    import tensorflow as tf
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False
    print("tf_keras not available. Inference cells will be skipped.")

try:
    import joblib
    import pandas as pd
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

JPG_PATH = "../Frontend/Images/shoulder_x-ray.jpg"
CNN_WEIGHTS  = None  # Path to the .h5 CNN checkpoint if available
ML_WEIGHTS   = None  # Path to the .joblib ML model if available
FEAT_PATH    = None  # Path to final_features.pkl if available

In [ ]:
# Load the test image and prepare two sizes used by the two modes
img_raw = cv2.imread(JPG_PATH, cv2.IMREAD_GRAYSCALE)
assert img_raw is not None, f"Could not read: {JPG_PATH}"

img_norm = img_raw.astype(np.float32)
img_norm = (img_norm - img_norm.min()) / (img_norm.max() - img_norm.min()) * 255
img_uint8 = img_norm.astype(np.uint8)

# Prepare inputs for each mode
def to_model_input(img_gray, size):
    resized = cv2.resize(img_gray, (size, size), interpolation=cv2.INTER_AREA)
    rgb     = cv2.cvtColor(resized, cv2.COLOR_GRAY2RGB).astype("float32") / 255.0
    return np.expand_dims(rgb, axis=0)  # (1, size, size, 3)

img_300 = to_model_input(img_uint8, 300)
img_512 = to_model_input(img_uint8, 512)

print(f"SEG_300 input shape : {img_300.shape}")
print(f"Fast_512 input shape: {img_512.shape}")

## 4. The Combined CNN Model

To avoid running the CNN twice (once for the probability, once for the features),
the production code builds a single `tf_keras.Model` with **two outputs**:

1. The final sigmoid output (classification probability)
2. The GlobalMaxPooling2D layer output (512-dimensional feature vector)

This combined model computes both in a single forward pass, halving the inference time
for the Fast_512 endpoint.

In [ ]:
cnn_model    = None
combined_cnn = None
ml_model     = None
selected_features = []

if TF_AVAILABLE and CNN_WEIGHTS and os.path.exists(CNN_WEIGHTS):
    cnn_model = tf_keras.models.load_model(CNN_WEIGHTS)

    # Locate the GlobalMaxPooling2D layer
    gmp_layer = cnn_model.get_layer("global_max_pooling2d")

    # Build combined model: one input, two outputs
    combined_cnn = tf_keras.Model(
        inputs=cnn_model.input,
        outputs=[cnn_model.output, gmp_layer.output],
    )
    print("CNN and combined model loaded.")
    print(f"CNN output shape        : {cnn_model.output.shape}")
    print(f"GMP feature vector shape: {gmp_layer.output.shape}")

if SKLEARN_AVAILABLE and ML_WEIGHTS and os.path.exists(ML_WEIGHTS):
    ml_model = joblib.load(ML_WEIGHTS)
    print("ML hybrid model loaded.")

if FEAT_PATH and os.path.exists(FEAT_PATH):
    selected_features = [str(f).strip() for f in joblib.load(FEAT_PATH)]
    print(f"Feature list loaded: {len(selected_features)} features expected by ML model.")

## 5. Mode A: SEG_300 Inference

In [ ]:
THRESHOLD = 0.5

if cnn_model is not None:
    # class_300_model is the VGG19 classifier trained at 300x300
    # Here we use cnn_model at 300x300 for demonstration
    prob_seg300 = float(cnn_model.predict(img_300, verbose=0)[0][0])
    diagnosis   = "Tendinopathy Detected" if prob_seg300 >= THRESHOLD else "No Tendinopathy Detected"

    print(f"SEG_300 probability : {prob_seg300:.4f}")
    print(f"SEG_300 diagnosis   : {diagnosis}")
else:
    print("Skipping SEG_300 inference (no model weights).")
    print("Expected output format:")
    print("  probability : float in [0, 1]")
    print("  diagnosis   : 'Tendinopathy Detected' or 'No Tendinopathy Detected'")

## 6. Mode B: Fast_512 with Hybrid Model

The hybrid model combines the 512-dimensional CNN feature vector with two clinical variables:

- **Age:** continuous integer (default 75 when not available from DICOM)
- **Sex:** binary (0 = female, 1 = male; default 0)

The feature names in the ML model's training data follow the convention `feature_0`,
`feature_1`, ..., `feature_511` for the CNN embeddings, plus `sex` and `edad` (age in Spanish).
The `final_features.pkl` file stores the subset of these 514 columns that were selected
during training (feature selection step).

The `reindex` call ensures that even if the CNN produces more or fewer features than expected,
the DataFrame passed to the ML model always has the exact columns it was trained on.

In [ ]:
DEFAULT_AGE = 75
DEFAULT_SEX = 0

# Simulate clinical metadata (in production, these come from DICOM tags)
patient_age = DEFAULT_AGE
patient_sex = DEFAULT_SEX

if combined_cnn is not None and ml_model is not None and selected_features:
    # Single forward pass: get probability AND feature vector
    prob_arr, feats = combined_cnn.predict(img_512, verbose=0)
    prob_cnn = float(prob_arr[0][0])

    if feats.ndim > 2:
        feats = feats.reshape(1, -1)

    # Build DataFrame with CNN feature column names
    col_names = [f"feature_{i}" for i in range(feats.shape[1])]
    df = pd.DataFrame(feats, columns=col_names)
    df["sex"]  = patient_sex
    df["edad"] = patient_age

    # Align to the exact feature set the ML model was trained on
    df_aligned = df.reindex(columns=selected_features, fill_value=0)

    prob_hyb = float(ml_model.predict_proba(df_aligned)[0][1])

    print(f"CNN probability    : {prob_cnn:.4f}")
    print(f"Hybrid probability : {prob_hyb:.4f}")
    print(f"Hybrid diagnosis   : {'Tendinopathy Detected' if prob_hyb >= THRESHOLD else 'No Tendinopathy Detected'}")
else:
    print("Skipping hybrid inference (no model weights).")
    print("Expected output format:")
    print("  prob_cnn  : float in [0, 1]  (pure CNN output)")
    print("  prob_hyb  : float in [0, 1]  (CNN features + age + sex -> ML model)")

## 7. Grad-CAM: Visualizing Model Attention

Gradient-weighted Class Activation Mapping (Grad-CAM, Selvaraju et al., 2017) produces a
coarse spatial map showing which regions of the input image most influenced the model's output.
In medical AI, this is valuable for clinical trust: a radiologist can verify that the model is
attending to the correct anatomical region rather than to background artifacts.

### How Grad-CAM works

1. **Select the target layer.** The last convolutional layer (`block5_conv4` in VGG19)
   produces feature maps that are spatially detailed enough to localize findings.

2. **Compute gradients.** Using a GradientTape, compute the gradient of the class score
   with respect to the target layer's output:
   `dScore / dConvOutput`

3. **Pool over spatial dimensions.** Average the gradients across height and width to get
   importance weights per channel: `alpha_k = mean(dScore / dConvOutput_k)`

4. **Weighted sum of activations.** Multiply each channel's activation map by its weight
   and sum: `heatmap = ReLU(sum_k(alpha_k * A_k))`

5. **Resize and overlay.** Upscale the heatmap to the input image size and overlay it
   as a colormap on the original X-ray.

The implementation splits the model into two sub-models at the `block5_conv4` layer to
allow the GradientTape to track gradients through the layer boundary.

In [ ]:
def build_gradcam_models(model):
    """
    Splits the model at block5_conv4 into:
      conv_model      - input -> last conv layer output
      classifier_model - conv output -> final score
    """
    if not TF_AVAILABLE:
        return None, None

    target_layer_name = "block5_conv4"
    base_model = model.get_layer("vgg19") if any(
        l.name == "vgg19" for l in model.layers
    ) else model

    conv_layer = None
    try:
        conv_layer = base_model.get_layer(target_layer_name)
    except Exception:
        for layer in reversed(base_model.layers):
            if "Conv2D" in layer.__class__.__name__:
                conv_layer = layer
                break

    if conv_layer is None:
        return None, None

    # Sub-model 1: original input -> conv output
    conv_model = tf_keras.models.Model(
        inputs=base_model.input,
        outputs=conv_layer.output
    )

    # Sub-model 2: conv output -> final score
    # Rebuilds the head (GlobalMaxPooling + Dense) using the trained weights
    inp = tf_keras.Input(shape=conv_model.output.shape[1:])
    gmp   = [l for l in model.layers if "GlobalMax" in l.__class__.__name__][0]
    dense = [l for l in model.layers if "Dense"     in l.__class__.__name__][-1]
    x = gmp(inp)
    x = tf_keras.layers.Dense(
        units=dense.units,
        weights=dense.get_weights()
    )(x)
    classifier_model = tf_keras.models.Model(inp, x)

    return conv_model, classifier_model


def compute_gradcam(conv_model, classifier_model, img_tensor, original_bgr):
    with tf.GradientTape() as tape:
        conv_out = conv_model(img_tensor)
        tape.watch(conv_out)
        score = classifier_model(conv_out)[:, 0]

    grads        = tape.gradient(score, conv_out)             # (1, H, W, C)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))      # (C,)

    conv_np   = conv_out.numpy()[0]                           # (H, W, C)
    pooled_np = pooled_grads.numpy()

    # Weight each feature map channel by its importance
    for i in range(pooled_np.shape[-1]):
        conv_np[:, :, i] *= pooled_np[i]

    heatmap = np.mean(conv_np, axis=-1)                       # (H, W)
    heatmap = np.maximum(heatmap, 0)                          # ReLU
    heatmap /= (heatmap.max() + 1e-10)                        # normalize
    heatmap  = 1.0 - heatmap                                  # invert if needed

    h_orig, w_orig = original_bgr.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w_orig, h_orig))
    heatmap_uint8   = np.uint8(255 * heatmap_resized)
    heatmap_color   = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    overlay = cv2.addWeighted(heatmap_color, 0.4, original_bgr, 1.0, 0)
    return overlay, heatmap_resized

In [ ]:
if TF_AVAILABLE and cnn_model is not None:
    conv_model, classifier_model = build_gradcam_models(cnn_model)

    if conv_model is not None:
        # Prepare background image (BGR, same size as model input)
        img_512_display = cv2.resize(img_uint8, (512, 512))
        bg_bgr = cv2.cvtColor(img_512_display, cv2.COLOR_GRAY2BGR)

        overlay, heatmap = compute_gradcam(conv_model, classifier_model, img_512, bg_bgr)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(img_512_display, cmap="gray")
        axes[0].set_title("Input (512x512)")
        axes[0].axis("off")

        axes[1].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
        axes[1].set_title("Grad-CAM heatmap")
        axes[1].axis("off")

        axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[2].set_title("Overlay on X-ray")
        axes[2].axis("off")

        plt.suptitle("Grad-CAM: regions influencing the tendinopathy prediction", fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print("Could not build Grad-CAM sub-models (layer not found).")
else:
    print("Skipping Grad-CAM visualization (no model weights).")
    print("The heatmap should highlight the supraspinatus tendon insertion area.")

## 8. Training Context (Conceptual)

The models were trained on a dataset of shoulder AP X-rays with binary labels
(tendinopathy / no tendinopathy). The training process followed these steps:

### CNN training
1. VGG19 with ImageNet weights was loaded with the top layers removed.
2. A GlobalMaxPooling2D + Dense(1, sigmoid) head was added.
3. The model was trained in two phases:
   - **Phase 1:** Freeze the VGG19 base, train only the head (fast convergence).
   - **Phase 2:** Unfreeze the last convolutional block (block5), fine-tune end-to-end
     with a small learning rate to avoid destroying pretrained features.
4. Data augmentation (horizontal flip, brightness/contrast jitter, rotation) was applied
   to mitigate the limited dataset size.

### Hybrid model training
1. The CNN was fixed after fine-tuning.
2. Feature vectors were extracted from the GlobalMaxPooling layer for all training images.
3. Age and sex were appended as additional columns.
4. A feature selection step (e.g., variance threshold or recursive elimination) reduced the
   514-dimensional space to a smaller set stored in `final_features.pkl`.
5. A scikit-learn pipeline (scaler + classifier) was trained on the selected features.
   The pipeline is serialized with `joblib.dump()` into the `.joblib` file.

### Evaluation
Both models were evaluated on a held-out test set using AUC-ROC, sensitivity, specificity,
and the confusion matrix at the 0.5 threshold.

## 9. Summary

| Component | Role | Input size | Output |
|-----------|------|-----------|--------|
| VGG19 CNN (class_300_model) | SEG_300 classifier | 300x300 | probability |
| VGG19 CNN (cnn_model) | Fast_512 feature extractor + classifier | 512x512 | probability + 512-dim vector |
| Hybrid ML model | Combines CNN features with clinical variables | 512+2 cols | probability |
| Grad-CAM | Spatial explanation of CNN decision | 512x512 | attention heatmap |

The next notebook (04) covers the second diagnostic pathway: detecting specific anatomical
regions with YOLO and then running a dedicated calcium deposit segmentation and classification
pipeline on those regions.